In [ ]:
import meta
import load
import analysis

import preprocess
import numpy as np
from collections import defaultdict

In [ ]:
records = []

dataset = meta.datasets[0]
records.extend(load.load_dataset(dataset))

# dateset = meta.datasets[1]
# records.extend(load.load_dataset(dateset))

In [ ]:
auxnote_override_map = defaultdict(lambda: "Other")
auxnote_override_map['(AFIB'] = "AF"
auxnote_override_map['(AFL'] = "AF"
auxnote_label_map = {"AF": 1, "Other": 0}

analysis.print_records_annotations(records)
records = preprocess.remove_auxnote_by_identifier(records, start_id='(V')  # remove (V events
analysis.print_records_annotations(records)
records = preprocess.override_auxnote(records, auxnote_override_map)
analysis.print_records_annotations(records)



## Preprocess Data

In [ ]:
records = preprocess.preprocess_records(records, frequency_resample=True, bandpass_filter=True)

In [ ]:
window_duration = 20  # seconds
stride_duration = 5  # seconds

samples = preprocess.generate_samples_from_records(records, window_duration=window_duration, stride_duration=stride_duration, auxnote_label_map = auxnote_label_map)
X_data, Y_data = preprocess.generate_inputs_from_samples(samples)
X, Y = X_data.copy(), Y_data.copy()

## Sanity Check (X, Y)

In [ ]:
# map Y matrix (n_sample, n_seconds) to majority label vecotr (n_sample,)
def get_majority_label(Y):
    Y_majority = []
    for i in range(len(Y)):
        Y_majority.append(np.bincount(Y[i]).argmax())
    return np.array(Y_majority)


# count the number of occurrences of each label
from collections import Counter
counter = Counter(Y.reshape(-1))
print("Total label in Y counted as", counter)

Y_majority = get_majority_label(Y)
print("Majority label in Y counted as", Counter(Y_majority))

samples_with_mixed_labels = []
for i in range(len(Y)):
    if len(set(Y[i])) > 1:
        samples_with_mixed_labels.append(i)
print(f"Samples with mixed labels {len(samples_with_mixed_labels)} out of {len(Y)}, percentage {len(samples_with_mixed_labels) / len(Y) * 100:.2f}%")

## Prepare for Training

In [ ]:
from preprocess import class_to_onehot, onehot_to_class
print("Before label conversion Y shape:", Y.shape)
Y = class_to_onehot(Y, num_classes=3)
print("After label conversion Y shape:", Y.shape)

In [ ]:
# # split the X,y data into train and test sets using stratified sampling on Y
# from sklearn.model_selection import train_test_split
# SEED = 42
# X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y_majority, random_state=SEED) # stratified sampling on Y_majority

# from sklearn.utils.class_weight import compute_class_weight
# class_weights = dict(enumerate(compute_class_weight(class_weight='balanced', classes=np.unique(Y_majority), y=Y_majority)))  
# class_weights = {k: v / sum(class_weights.values()) for k, v in class_weights.items()} # normalize the weights to sum to 1
# print("Class Weights:", class_weights)

In [ ]:
# sample equal number of samples for each class based on Y_majority
from sklearn.utils import resample
SEED = 42
n_samples = 325
X_resampled, Y_resampled = [], []
for label in np.unique(Y_majority):
    X_label = X[Y_majority == label]
    Y_label = Y[Y_majority == label]
    X_resampled_label, Y_resampled_label = resample(X_label, Y_label, n_samples=n_samples, random_state=SEED) # resample to 1000 samples for each label
    X_resampled.append(X_resampled_label)
    Y_resampled.append(Y_resampled_label)
X_resampled = np.concatenate(X_resampled, axis=0)
Y_resampled = np.concatenate(Y_resampled, axis=0)
print("X_resampled shape:", X_resampled.shape)

# split the X, Y data into train and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X_resampled, Y_resampled, test_size=0.2, random_state=SEED) # stratified sampling on Y_majority
class_weights = {0:1/3, 1:1/3, 2:1/3} # normalize the weights to sum to 1
print("Class Weights:", class_weights)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv1D, GlobalAveragePooling1D, Dropout, BatchNormalization, ReLU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf

def network(X_train, y_train, X_test, y_test,
             unit_size, class_size, class_weights,
             learning_rate=0.001, batch_size = 32, epochs=40, 
             loss_type='cross_entropy', modelCheckPoint=False):    
    ####################################################################
    output_size = unit_size * class_size  # output size is the number of classes times the number of units
    print("Class Weights:", class_weights)
    
    def custom_loss(y_true, y_pred):
        if class_size != len(class_weights):
            raise ValueError("The length of class_weights must match the number of classes.")
        
        # Reshape tensors
        y_true = tf.reshape(y_true, [-1, unit_size, class_size])
        y_pred = tf.reshape(y_pred, [-1, unit_size, class_size])

        loss = 0.0
        if loss_type == 'cross_entropy':
            for i in range(len(class_weights)):
                if class_weights[i] > 0:
                    loss += class_weights[i] * tf.reduce_mean(
                        -y_true[:, :, i] * tf.math.log(y_pred[:, :, i] + 1e-15) # average loss across all samples and units
                    )
        elif loss_type == 'mse':
            for i in range(len(class_weights)):
                if class_weights[i] > 0:
                    loss += class_weights[i] * tf.reduce_mean(
                        tf.square(y_true[:, :, i] - y_pred[:, :, i])  # average loss across all samples and units
                    )
        else:
            raise ValueError("Invalid loss type. Use 'cross_entropy' or 'mse'.")
        return loss

    def custom_accuracy(y_true, y_pred):
        y_true = tf.reshape(y_true, [-1, unit_size, class_size])
        y_pred = tf.reshape(y_pred, [-1, unit_size, class_size])
        correct_predictions = tf.equal(tf.argmax(y_true, axis=2), tf.argmax(y_pred, axis=2))
        return tf.reduce_mean(tf.cast(correct_predictions, tf.float32)) # average accuracy across all samples and units

    ####################################################################
    inputs_cnn = Input(shape=(X_train.shape[1], 1))

    # Block 1
    x = Conv1D(16, kernel_size=7, padding='same')(inputs_cnn)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # Block 2
    x = Conv1D(32, kernel_size=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(0.2)(x)
    x = Conv1D(32, kernel_size=3, padding='same')(x)

    # Block 3
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(0.2)(x)
    x = Conv1D(64, kernel_size=3, padding='same')(x)

    # Final block
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = GlobalAveragePooling1D()(x)

    # Output layer
    outputs_cnn = Dense(output_size, activation='softmax')(x)

    optimizer = Adam(learning_rate=learning_rate)
    model = Model(inputs=inputs_cnn, outputs=outputs_cnn)
    model.compile(optimizer=optimizer, loss=custom_loss, metrics=[custom_accuracy])

    # define callbacks for early stopping and saving the best model with lowest validation loss
    callbacks = [EarlyStopping(monitor='val_loss', patience=8)]
    if modelCheckPoint:
        callbacks.append(ModelCheckpoint(filepath='best_model.h5', monitor='val_loss', save_best_only=True))
    history=model.fit(X_train, y_train, epochs=epochs, 
                      callbacks=callbacks, batch_size=batch_size, validation_data=(X_test, y_test),
                      #class_weight=class_weights # assign class weights to the model 
                      ) 
    if modelCheckPoint:
        model.load_weights('best_model.h5')
    return (model, history)

In [ ]:
model, history=network(X_train, Y_train, X_test, Y_test,
                      unit_size=window_duration, class_size=3, class_weights=class_weights,
                      learning_rate=0.0001, batch_size=32, epochs=100, loss_type='cross_entropy', modelCheckPoint=False)

### Error Analysis

In [ ]:
from analysis import plot_accuracy_and_loss, plot_confusion_matrix
from sklearn.metrics import confusion_matrix

In [ ]:
import matplotlib.pyplot as plt
def plot_accuracy_and_loss(history, X_test, y_test, model, accuracy_keys = ['custom_accuracy', 'val_custom_accuracy'], loss_keys = ['loss', 'val_loss']):
    scores = model.evaluate((X_test), y_test, verbose=0)
    print("Accuracy: %.2f%%" % (scores[1]*100))
    
    print(history)
    fig1, ax_acc = plt.subplots()
    plt.plot(history.history[accuracy_keys[0]])
    plt.plot(history.history[accuracy_keys[1]])
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Model - Accuracy')
    plt.legend(['Training', 'Validation'], loc='lower right')
    plt.show()
    
    fig2, ax_loss = plt.subplots()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Model- Loss')
    plt.legend(['Training', 'Validation'], loc='upper right')
    plt.plot(history.history[loss_keys[0]])
    plt.plot(history.history[loss_keys[1]])
    plt.show()


In [ ]:
plot_accuracy_and_loss(history, X_test, Y_test, model)

In [ ]:
Y_pred_onehot = model.predict(X_test)
Y_pred_class = onehot_to_class(Y_pred_onehot, num_classes=3)
Y_test_onehot = Y_test.copy()
Y_test_class = onehot_to_class(Y_test_onehot, num_classes=3)

# calculate the accuracy for each class
for i in range(3):
    acc = np.sum((Y_test_class == i) & (Y_pred_class == i)) / np.sum(Y_test_class == i)
    print(f"Accuracy for class {i}: {acc:.2f}")

In [ ]:
# calculate the confusion matrix based on Y_test_class and Y_pred_class
C = np.zeros((3, 3))
for i in range(len(Y_test_class)):
    for j in range(len(Y_test_class[i])):
        C[int(Y_test_class[i][j]), int(Y_pred_class[i][j])] += 1
print("Confusion Matrix:")
print(C)

In [ ]:
y_pred=model.predict(X_test)
cnf_matrix = confusion_matrix(Y_test, y_pred.argmax(axis=1))
plot_confusion_matrix(cnf_matrix, classes=['N', 'AFib', 'AFlu'], normalize=True)